# Data Acquistion
Using the Utah State legislative API, pulling the data on bills. Thank you to Lucy Ballou for writing this code.

In [ ]:
# importing packages
import requests
import pandas as pd
import json
import time
import numpy as np
import re

In [ ]:
# pulling data from the legislative API [provided by Lucy Ballou]

TOKEN = "redacted"
BASE_URL = "https://glen.le.utah.gov/bills"

# defining sessions (pre-2016 is unavail with API)
sessions = [
    "2016GS", "2017GS", "2018GS", "2019GS", "2020GS",
    "2021GS", "2022GS", "2023GS", "2024GS", "2025GS", "2026GS"
]

# extracting bill numbers
all_bills = {}

for session in sessions:
    url = f"{BASE_URL}/{session}/billlist/{TOKEN}"
    response = requests.get(url)

    if response.status_code == 200:
        try:
            data = response.json()

            # handling the fact that some years are formatted as plain lists, some are objects, some are strings
            if isinstance(data, list):
                bill_list = data
            elif isinstance(data, dict) and "bills" in data:
                bill_list = data["bills"]
            else:
                print(f"{session}: Unexpected format")
                continue

            bill_numbers = [bill['number'] for bill in bill_list]
            all_bills[session] = bill_numbers
            print(f"{session}: {len(bill_numbers)} bills")
        except Exception as e:
            print(f"{session}: Failed - {e}")
    else:
        print(f"{session}: ERROR {response.status_code}")

    time.sleep(1)

total = sum(len(bills) for bills in all_bills.values())
print(f"\nTotal bills across sessions: {total}")

# actual extracting
def extract_bill_data(session, bill_number, token):
    url = f"{BASE_URL}/{session}/{bill_number}/{token}"
    response = requests.get(url)

    if response.status_code != 200:
        return None

    try:
        data = response.json()
    except:
        return None

    if isinstance(data, dict) and "bill" in data and isinstance(data["bill"], dict):
        data = data["bill"]

    if isinstance(data, str):
        try:
            data = json.loads(data)
        except:
            return None

    # skip if still not a dict
    if not isinstance(data, dict):
        return None

    is_old_format = "shorttitle" in data

    if is_old_format:
        # old format
        subjects = data.get("subjects", [])

        # committees from agendas
        agendas = data.get("agendas", [])
        committees = list(set(
            a.get("committee", a.get("committeeName", ""))
            for a in agendas
        )) if agendas else []

        # originating chamber from bill number prefix
        bill_id = data.get("bill", "")
        if bill_id.startswith("HB") or bill_id.startswith("HJ"):
            orig_chamber = "House"
        elif bill_id.startswith("SB") or bill_id.startswith("SJ"):
            orig_chamber = "Senate"
        else:
            orig_chamber = ""

        return {
            "bill_id": bill_id,
            "short_title": data.get("shorttitle", ""),
            "sponsor": data.get("sponsor", ""),
            "co_sponsors": "None",
            "floor_sponsor": data.get("floorsponsor", ""),
            "last_action": data.get("lastaction", ""),
            "last_action_owner": data.get("lastactionowner", ""),
            "subjects": "; ".join(subjects) if isinstance(subjects, list) else str(subjects),
            "num_substitutes": 0,
            "substitute_sponsors": "None",
            "active_version": data.get("version", ""),
            "subjects_changed": False,
            "committees": "; ".join(committees),
            "money_appropriated": data.get("monies", ""),
            "originating_chamber": orig_chamber,
            "final_chamber": "",
            "sponsor_id": data.get("sponsor", ""),
            "session_year": session[:4],
            "session_id": session,
        }

    else:
        # new format
        versions = data.get("billVersionList", [])

        # finding and extracting active version for co-sponsors and subjects
        active_version = None
        active_version_label = ""
        for version in versions:
            if version.get("activeVersion", False):
                active_version = version
                active_version_label = version.get("billNumber", "")
                break
        if active_version is None and versions:
            active_version = versions[-1]
            active_version_label = versions[-1].get("billNumber", "")

        subjects = []
        if active_version:
            subjects = list(set(
                s["description"] for s in active_version.get("subjectList", [])
            ))

        co_sponsors = []
        if active_version:
            co_sponsors = [
                cs.get("name", "") for cs in active_version.get("coSponsorList", [])
            ]

        # substitute stuff
        substitutes = [v for v in versions if int(v.get("subVersion", 0)) > 0]
        num_substitutes = len(substitutes)

        sub_sponsors = "; ".join(
            f"{v['billNumber']}: {v.get('versionSponsorName', 'Unknown')}"
            for v in substitutes
        )

        # whether subjects changed across versions?
        def get_subjects_set(version):
            return set(s["description"] for s in version.get("subjectList", []))

        if len(versions) > 1:
            original_subjects = get_subjects_set(versions[0])
            subjects_changed = any(
                get_subjects_set(v) != original_subjects for v in versions[1:]
            )
        else:
            subjects_changed = False

        # getting committees from agendaList
        committees = list(set(
            a["committeeName"] for a in data.get("agendaList", [])
        ))

        # finding original and final chamber from action history
        originating_chamber = data.get("primeSponsorHouse", "")
        final_chamber = ""
        for action in reversed(data.get("actionHistoryList", [])):
            if action.get("actionClass") in ("H", "S"):
                final_chamber = "House" if action["actionClass"] == "H" else "Senate"
                break

        return {
            "bill_id": data.get("billNumber", ""),
            "short_title": data.get("shortTitle", ""),
            "sponsor": data.get("primeSponsorName", ""),
            "co_sponsors": "; ".join(co_sponsors) if co_sponsors else "None",
            "floor_sponsor": data.get("floorSponsorName", ""),
            "last_action": data.get("lastAction", ""),
            "last_action_owner": data.get("lastActionOwner", ""),
            "subjects": "; ".join(subjects),
            "num_substitutes": num_substitutes,
            "substitute_sponsors": sub_sponsors if sub_sponsors else "None",
            "active_version": active_version_label,
            "subjects_changed": subjects_changed,
            "committees": "; ".join(committees),
            "money_appropriated": data.get("moniesAppropriated", ""),
            "originating_chamber": "House" if originating_chamber == "H" else "Senate",
            "final_chamber": final_chamber,
            "sponsor_id": data.get("primeSponsor", ""),
            "session_year": data.get("year", ""),
            "session_id": data.get("sessionID", ""),
        }


# extraction loop
all_rows = []

for session, bill_numbers in all_bills.items():
    print(f"\n--- {session}: {len(bill_numbers)} bills ---")

    for i, bill_number in enumerate(bill_numbers):
        result = extract_bill_data(session, bill_number, TOKEN)

        if result:
            all_rows.append(result)
        else:
            print(f"  Failed: {bill_number}")

        # progress update every 50 bills
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(bill_numbers)}")

        time.sleep(1)

    # save progress after each session
    df_progress = pd.DataFrame(all_rows)
    df_progress.to_csv("utah_bills_progress.csv", index=False)
    print(f"  Saved progress: {len(all_rows)} total rows so far")

# build dataframe
df = pd.DataFrame(all_rows)
df.to_csv("utah_bills_final.csv", index=False)
print(f"\nFinal dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())
